# Notebook Roadmap
00_driver_2.0 was a monolithic driver notebook built around a single question: do the cyanobacteria in Mammoth Cave encode KaiABC, the circadian clock? It was written almost entirely around KY-Mam-100624-C as a pilot sample, with KY-Mam-B as an afterthought. The notebook served dual purposes — it launched the Snakemake pipeline (QC, assembly, polish, taxonomy, binning) via subprocess, and then immediately dove into the most specific hypothesis available at the time. Cyano contigs were filtered from Kraken2 output, Prodigal predicted their genes, all 75,000+ proteins from the full assembly were run through HMMER against KaiABC profiles, and the negative result (no hits) closed the loop. The analysis never really left KY-Mam-C, and the downstream sections on 16S, phylogenetics, and batch processing were mostly templates and notes pointing toward work that hadn't happened yet.

With very low coverage of the cyanobacteria genomes present, the focus shifted away from a single gene family to a broader question: what is the full cyanobacterial community across these ten lake and cave samples. That reframing demanded a different architecture.

01_community_analysis inherited the pilot infrastructure from 00 — the run_cmd() wrapper, the Kraken2 parsing logic, the barrnap/BLAST/MAFFT/FastTree chain — but reoriented it around community comparison rather than gene-level hypothesis testing. The B vs. C cave comparison became the prototype for a workflow that was always intended to scale.

02_all_samples_community is the scaled version: the same SILVA BLASTn + phylogenetics framework applied across all ten samples simultaneously, with the contaminant profiling (bacteria, fungi, algae) added as a way to characterize the non-target community before isolating the cyanobacterial signal.

03a_contig_id_gene_context closes the loop on the most interesting finding from 02 — the sequences with no close SILVA match — and in doing so revisits the gene-level analysis that 00 pioneered (Prodigal, BLASTp, genomic context), but this time in service of novelty characterization rather than a specific clock-gene hypothesis. In that sense, 03 is 00 grown up: the same instinct to look at genes, applied with a more open-ended question.

# 01 Metagenome Command Center

This notebook is an accessible command center for the Snakemake metagenomics pipeline.

It is designed to be:
- **Intuitive**: each section explains *why* it exists and what to look for.
- **Shareable**: paths are detected automatically when possible.

## What this notebook covers
1. Project setup (paths, environment)
2. Loading/validating `config/samples.tsv`
3. Sanity checks on FASTQ inputs
4. Medaka model configuration
5. Running Snakemake (dry-run and real runs)
6. Full-assembly gene prediction with Pyrodigal
7. KaiABC circadian clock gene search (HMMER)
8. Coverage analysis of cyanobacterial contigs
9. Findings & interpretation — KY-Mam-100624-C (Mammoth Cave)
10. Next steps: batch pipeline for remaining 9 samples
11. Reference genome mapping & KaiABC absence (3-method proof)
12. KY-Mam-B comparison (outside cave)
13. 16S rRNA phylogenetics & species novelty
14. Genome completeness & FastANI

> Tip: If you want to focus on a subset of samples, prefer **environment variables** (e.g., `FocusSAMPLES=...`) rather than editing rules.


## 0) Setup: imports and project root

This cell detects the project directory by searching upward for a `Snakefile`.

In [1]:
import os, sys, platform, subprocess, shlex
from pathlib import Path
import pandas as pd
import yaml

def find_project_root(start=None):
    """Find the directory containing the Snakefile by walking up from `start` (or CWD)."""
    p = (start or Path.cwd()).resolve()
    for parent in [p] + list(p.parents):
        if (parent / "Snakefile").exists():
            return parent
    raise FileNotFoundError("No Snakefile found in this directory or any parent directory.")

PROJECT = find_project_root()
print("Project root:", PROJECT)
print("Python:", sys.version.split()[0], "| Arch:", platform.machine(), "| OS:", platform.system())
print("Conda prefix:", os.environ.get("CONDA_PREFIX", "(none)"))


Project root: /Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria/GenomicAnalyses/MetagenomicAnalyses/metagenome
Python: 3.11.14 | Arch: arm64 | OS: Darwin
Conda prefix: /Users/jaemac/miniforge3/envs/meta-ont-macos-arm64


## 1) Load and preview `config/samples.tsv`

**Goal:** make sure we know *exactly* what sample IDs exist and where the FASTQs live.

This pipeline expects a tab-separated file at `config/samples.tsv` with (at minimum):
- a sample identifier column (commonly `ID` or `sample_id`)
- a FASTQ path column (commonly `DIRfastQ` or `fastq`)

This cell loads the table and normalizes to two columns:
- `sample`
- `fastq`


In [2]:
samples_path = PROJECT / "config/samples.tsv"
assert samples_path.exists(), f"Missing: {samples_path}"

raw_samples = pd.read_csv(samples_path, sep="\t")
display(raw_samples)

def normalize_samples_table(df: pd.DataFrame) -> pd.DataFrame:
    # Common column name variants
    id_cols = [c for c in ["ID", "sample_id", "sample", "Sample", "name"] if c in df.columns]
    fq_cols = [c for c in ["DIRfastQ", "fastq", "FASTQ", "reads", "path"] if c in df.columns]
    if not id_cols:
        raise ValueError(f"No sample id column found. Columns: {list(df.columns)}")
    if not fq_cols:
        raise ValueError(f"No fastq path column found. Columns: {list(df.columns)}")

    out = df.copy()
    out = out.rename(columns={id_cols[0]: "sample", fq_cols[0]: "fastq"})
    out["sample"] = out["sample"].astype(str)
    out["fastq"] = out["fastq"].astype(str)
    return out[["sample", "fastq"]]

samples = normalize_samples_table(raw_samples)
print("Loaded samples:", len(samples))


,ID,DIRfastQ
0,MN-Por-091024-A,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...
1,TN-Cla-092324-E,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...
2,WI-Bar-090824-B,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...
3,NC-Che-092424-E,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...
4,KY-Mam-100624-B,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...
5,KY-Mam-100624-C,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...
6,GA-Cor-092624-B,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...
7,FL-Wim-100124-B,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...
8,MN-Col-091924-B,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...
9,FL-Hom-052325-G,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...


Loaded samples: 13


## 2) Sanity-check FASTQ paths and rough file sizes

**Why:** many downstream failures are just bad paths or missing files.

This cell checks:
- Does the file exist?
- What is its size on disk? (useful to spot suspiciously small files)

> Note: file size is not read depth, but it is a quick red-flag detector.


In [3]:
def human_bytes(n: int) -> str:
    for unit in ["B","KB","MB","GB","TB","PB"]:
        if n < 1024:
            return f"{n:.1f} {unit}"
        n /= 1024
    return f"{n:.1f} EB"

rows = []
for _, row in samples.iterrows():
    sid = row["sample"]
    fq = Path(row["fastq"]).expanduser()
    exists = fq.exists()
    size = fq.stat().st_size if exists else 0
    rows.append({"sample": sid, "fastq": str(fq), "exists": exists, "size": human_bytes(size)})

qc_df = pd.DataFrame(rows).sort_values(["exists","size"], ascending=[False, False])
display(qc_df)

missing = qc_df[~qc_df["exists"]]
if len(missing):
    print("\nMissing FASTQs (fix these first):")
    display(missing)


,sample,fastq,exists,size
4,KY-Mam-100624-B,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...,True,987.9 MB
2,WI-Bar-090824-B,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...,True,950.7 MB
5,KY-Mam-100624-C,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...,True,938.5 MB
9,FL-Hom-052325-G,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...,True,87.5 MB
0,MN-Por-091024-A,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...,True,792.1 MB
11,KY-Mam-100624-C2,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...,True,3.6 GB
10,WI-Bar-090824-B2,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...,True,3.3 GB
12,CA-Mec-092525-D,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...,True,3.3 GB
8,MN-Col-091924-B,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...,True,1020.6 MB
7,FL-Wim-100124-B,/Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria...,True,1.9 GB


## Terms
`[1] **Oxford Nanopore Technologies** (ONT)` is a 3rd generation sequencing technology that directly analyzes DNA or RNA as they pass through a biological   
pore embedded in an electrical membrane.  

`[2] **read:**` A sequencing read is the raw data output, a text string of bases (A,T,C,G), which represent a short segment of DNA. With long-read technology,  
reads can be > 10,000 base pairs (bp) long.

`[3] **consensus sequence:**` A representative sequence derived from aligning multiple raw sequence reads or related sequences.

`[4] **contig:**` Derived from the word *contig*uous and is a continuous, uninterrupted *consensus* sequence assembled from overlapping, shorter fragments.



## 3) Polishing with Racon and Medaka

### What "polishing" means in this pipeline?  
After long-read assembly, the contigs are usually **structurally correct** (good contiguity) but still contain **consensus-level base errors** (mismatches/indels),
especially around homopolymers* (more common in eukaryotes) and other systematic ONT error modes.  
**Polishing:** maps reads back to the draft assembly and using the multiple reads as evidence to improve the consensus sequence (high per-base accuracy). This matters for 
downstream gene calling, taxonomy, and binning because small indel/mismatch erros can break ORFs** and distort sequence similarity.

***homopolymers:** DNA or RNA sequences consisting of long, consecutive stretches of a single, repeating nucleotide base. Serve regulatory roles but cause sequencing woes.

****ORFs:** An open reading frame (ORF) is a continuous stretch of DNA or RNA sequence that contains a start codon, followed by triplet codons encoding amino acids, and ends with a stop codon. It represents a potential protein-coding region, serving as a fundamental tool for gene prediction and functional analysis in genomics. 

### Why Racon *and then* Medaka
- **Racon** is a fast, alignment-based consensus corrector designed to improve draft contigs produced by fast long-read assemblers (like miniasm/Flye-style pipelines). It's a solid "first-pass" that can quickly reduce the error rate. 
- **Medaka** is ONT's neural-network consensus polisher: it applies a trained model to the *read pileup against a draft assembly* to produce a higher-accuracy consensus, and it's explicitly designed for nanopore data, like ours.  
**In other words:** Racon is a strong "coarse correction" stage; Medaka is the "ONT-specialized refinement" stage.

### Why is Medaka a good choice for ONT + Flye
- Medaka is built specifically for nanopore consensus polishing using neural nets on aligned reads.  
- The Medaka project explicitly notes it can outperform graph-based/basecall-only polishing approaches (Racon is the classic example).  
- Medaka has been trained with Flye-style draft assemblies in mind, which matches our assembly path.

### Why the Medaka model matters
Medaka’s accuracy depends on using a model that matches your **basecaller + chemistry**. The official docs emphasize:
- “For best results it is important to specify the correct inference model, according to the basecaller used.”
- You can list valid models in terminal with `medaka tools list_models`.
- Newer basecallers often annotate outputs with their model; Medaka can try to auto-select, and you can inspect what it would choose.

### How to choose a Medaka model
**In Terminal (within the project directory): **  
- Get information from data files:
> % zcat <  results/02_filter/SAMPLE_ID.flt.fq.gz 
- Let Medaka choose:
> % medaka tools resolve_model --auto_model consensus results/02_filter/KY-Mam-100624-C.flt.fq.gz   
> % medaka tools resolve_model --auto_model consensus_bacteria results/02_filter/KY-Mam-100624-C.flt.fq.gz



**Sources:**  
Medaka (ONT) README / models / Flye notes:  https://github.com/nanoporetech/medaka    
Racon polishing overview:  https://denbi-long-read-training.readthedocs.io/en/latest/polishing/medaka/racon.html

## 3a) Medaka Model Configuration (polishing)

### How we encode this in the pipeline
We store a default model and (optionally) per-sample overrides in `config/medaka_models.yaml` so we can be explicit and reproducible when:
- basecaller versions differ across runs/samples
- we re-basecall data (which can change the recommended Medaka model)

The pipeline reads `config/medaka_models.yaml`, typically with:
- `default_model`
- optional per-sample overrides under `models:`

We enable Medaka’s bacterial/methylation model (--bacteria) for native metagenomic ONT reads because it better matches bacterial DNA modifications and can improve  
consensus accuracy when compatible with the basecaller model. This flag can be modified in the medak yaml file by changing medaka:bacteria: to false.

In [4]:
cfg_path = PROJECT / "config" / "medaka_models.yml"
assert cfg_path.exists(), f"Missing: {cfg_path}"

with open(cfg_path) as f:
    cfg = yaml.safe_load(f) or {}

print("Default model:", cfg.get("default_model"))
print("Overrides:", cfg.get("models") or {})
print("Bacteria flag:", (cfg.get("medaka") or {}).get("bacteria", False))
# display(pd.DataFrame(sorted((cfg.get("models") or {}).items()), columns=["sample","model"]))

Default model: r1041_e82_400bps_sup_v5
Overrides: None
Bacteria flag: True


## 4) Snakemake runner helpers

**Goal:** run Snakemake from the notebook in a consistent, debuggable way.

In [5]:
def run_cmd(cmd, cwd=PROJECT, env=None):
    """Run a command and stream output (raises if it fails)."""
    if isinstance(cmd, str):
        printable = cmd
    else:
        printable = " ".join(shlex.quote(str(x)) for x in cmd)
    print("\n$ " + printable)

    proc_env = os.environ.copy()
    if env:
        proc_env.update({k: str(v) for k, v in env.items()})

    p = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        env=proc_env,
        shell=isinstance(cmd, str),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {p.returncode}")

def snakemake(
    targets,
    cores=4,
    snakefile="Snakefile",
    dry_run=False,
    use_conda=False,      # set True only if you want Snakemake to manage envs
    rerun_incomplete=True,
    printshellcmds=True,
    extra=None,
    env=None,
    focus_samples=None,     # list/tuple or comma-string
    cov_read_samples=None  # list/tuple or comma-string
# Caution (scales as targets × read-sets): Coverage mapping creates one BAM per (target contigs s) × (read sample t). 
# If you focus TARGET_SAMPLES to 1 but leave COV_READ_SAMPLES as all, you’ll run a 1×N mapping job set (useful for 
# differential coverage; expensive if N is large).
):
    """Convenience wrapper for running Snakemake."""
    cmd = ["snakemake", "-s", snakefile, "-j", str(cores)]
    if dry_run:
        cmd.append("-n")
    if use_conda:
        cmd.append("--use-conda")
    if rerun_incomplete:
        cmd.append("--rerun-incomplete")
    if printshellcmds:
        cmd.append("--printshellcmds")
    if extra:
        cmd += list(extra)
    cmd += list(targets) if isinstance(targets, (list, tuple)) else [targets]

    env2 = dict(env or {})
    if focus_samples:
        env2["FocusSAMPLES"] = ",".join(focus_samples) if isinstance(focus_samples, (list,tuple)) else str(focus_samples)
    if cov_read_samples: 
        env2["COV_READ_SAMPLES"] = ",".join(cov_read_samples) if isinstance(cov_read_samples, (list, tuple)) else str(cov_read_samples)
        
    print("DEBUG env2:", {k: env2.get(k) for k in ["FocusSAMPLES","COV_READ_SAMPLES"]})

    # IMPORTANT: run Snakemake exactly once, after env overrides are assembled
    run_cmd(cmd, env=env2) 
    
    print("Environmental overrides:", {k: env2[k] for k in ("FocusSAMPLES","COV_READ_SAMPLES") if k in env2})


print("Ready. Example (dry-run all):")
print('snakemake(["all"], cores=4, dry_run=True)')

Ready. Example (dry-run all):
snakemake(["all"], cores=4, dry_run=True)


### 4a) Example dry-runs (safe to copy/paste)

Uncomment and run one at a time.

**Dry-run the whole workflow (focused):**  
- Focus targets to one sample
- Compute depth table for that sample


In [39]:
# Example: dry-run a single concrete target file
# env = {"FocusSAMPLES": "KY-Mam-100624-C", "LOCK_UPSTREAM": "1", "COV_READ_SAMPLES": "KY-Mam-100624-C"}
# snakemake(["results/06_cov/KY-Mam-100624-C/depth.tsv"], cores=4, dry_run=True, env=env)

## 5) Run it for Real! 

### 5a) RUN CONFIGURATION 

#### Key environment controls used by this pipeline
- `focus_samples`  
  Restrict *targets* (assemblies / downstream outputs) to these samples.
- `LOCK_UPSTREAM=1`  
  Protect upstream outputs (QC/trim/filter/assembly/polish/map) from accidental overwrites.
- `cov_read_samples= ['S1','S2',...]`
  Control which read sets `{t}` are used to compute coverage profiles for binning.

> **Recommendation:** Use `focus_samples` for iterative development; keep `cov_read_samples` small at first,
> and expand to "neighbors" later once you have a data-driven grouping.
> We compute coverage using reads from the target sample and a nearby ecological neighbor; for example run with Mammouth cave samples 
> most contigs show zero cross-sample coverage, but a subset co-vary across samples, providing informative differential coverage for
> binning without introducing excess noise.


In [6]:
# ======= RUN CONFIGURATION ============================
# If you change this, you must rerun 'snakemake runner helpers'

# Which assemblies/contigs to process (targets) 
focus_samples = ['WI-Bar-090824-B2','KY-Mam-100624-C2', 'CA-Mec-092525-D']

# Which read sets to contribute to coverage estimation
# Include the target itself + nearby/related samples 
cov_read_samples = ['WI-Bar-090824-B2', 'KY-Mam-100624-C2', 'CA-Mec-092525-D'] 

print("focus_samples:", focus_samples)
print("cov_read_samples:", cov_read_samples)

focus_samples: ['WI-Bar-090824-B2', 'KY-Mam-100624-C2', 'CA-Mec-092525-D']
cov_read_samples: ['WI-Bar-090824-B2', 'KY-Mam-100624-C2', 'CA-Mec-092525-D']


In [ ]:
# ======= RUN ==========================================
# Set 'dry_run=False' to execute.
target = "results/10_tax_profile/KY-Mam-100624-C/contigs.kraken.report" # change to "all" to run full pipeline

snakemake(
    targets = "all",
    cores = 2,
    use_conda = True,
    dry_run = False,
    focus_samples = focus_samples,
    cov_read_samples = cov_read_samples
)

DEBUG env2: {'FocusSAMPLES': 'WI-Bar-090824-B2,KY-Mam-100624-C2,CA-Mec-092525-D', 'COV_READ_SAMPLES': 'WI-Bar-090824-B2,KY-Mam-100624-C2,CA-Mec-092525-D'}

$ snakemake -s Snakefile -j 2 --use-conda --rerun-incomplete --printshellcmds all
ENV FocusSAMPLES = WI-Bar-090824-B2,KY-Mam-100624-C2,CA-Mec-092525-D
ENV COV_READ_SAMPLES = WI-Bar-090824-B2,KY-Mam-100624-C2,CA-Mec-092525-D
TARGET_SAMPLES:   ['WI-Bar-090824-B2', 'KY-Mam-100624-C2', 'CA-Mec-092525-D']
COV_READ_SAMPLES: ['WI-Bar-090824-B2', 'KY-Mam-100624-C2', 'CA-Mec-092525-D']
Assuming unrestricted shared filesystem usage.
host: jMac.local
Building DAG of jobs...
Using shell: /bin/bash
Provided cores: 2
Rules claiming more threads will be scaled down.
Job stats:
job                       count
----------------------  -------
all                           1
assemble_flye                 3
bin_metabat2                  3
depth_table                   3
extract_16S                   3
filt                          3
idx_contigs        

---
# ⚠️ STOP HERE — Pipeline Complete

**Everything above this line is the pipeline.** Once the Snakemake run cell (Section 5) finishes without errors, your samples are fully processed and you can move on to `01_community_analysis.ipynb`.

---

## Optional: Single-Sample Exploratory Analysis

The cells below are **optional, ad hoc** analyses written around `KY-Mam-100624-C`. They are not part of the pipeline and do not need to be run.

If you want to use them for a different sample, update the `sample` variable at the top of each code cell to the ID you want (e.g. `KY-Mam-100624-C2`, `WI-Bar-090824-B2`, or `CA-Mec-092525-D`).

**Topics covered below:**
- Kraken2 report parsing / cyanobacteria subtree extraction
- MetaBAT2 bin inspection
- Prodigal / Pyrodigal gene prediction
- KaiABC HMM search
- Per-contig coverage from BAM

## Get all taxids in the Cyanobacteria subtree (robust)
### 1. Get Kraken Report

In [ ]:
import pandas as pd, re

# Load Kraken Report
rep = pd.read_csv(str(PROJECT / "results/10_tax_profile/KY-Mam-100624-C/contigs.kraken.report"), sep="\t", header=None, names=["pct","clade","taxon","rank", "taxid", "name"])
display(rep.head())

# Load .tsv file (use later)
df = pd.read_csv(str(PROJECT / "results/10_tax_profile/KY-Mam-100624-C/contigs.kraken.tsv"), sep="\t", header=None, names=["status","contig","taxid","length","lca_taxid","lca_name"])
display(df.head())

,pct,clade,taxon,rank,taxid,name
0,5.18,88,88,U,0,unclassified
1,94.82,1612,2,R,1,root
2,94.71,1610,22,R1,131567,cellular organisms
3,91.06,1548,40,D,2,Bacteria
4,68.65,1167,15,P,1224,Proteobacteria


,status,contig,taxid,length,lca_taxid,lca_name
0,C,contig_1,Porphyrobacter sp. CACIAM 03H1 (taxid 2003315),58344,0:72 911045:4 0:1 1224:14 0:1010 1896196:1 0:6...,NaN
1,C,contig_10,Pseudomonas mendocina NK-01 (taxid 1001585),65685,1038922:1 0:77 286:2 0:48 741155:5 286:8 0:5 2...,NaN
2,C,contig_100,Gemmatimonas phototrophica (taxid 1379270),13809,0:557 2:1 0:11 2:3 0:1087 1379270:5 0:3585 100...,NaN
3,C,contig_1000,Dyella japonica A8 (taxid 1217721),5300,0:437 675864:5 0:407 1217721:3 0:1469 1224:4 0...,NaN
4,C,contig_1001,Porphyrobacter sp. CACIAM 03H1 (taxid 2003315),95665,0:40 2051553:5 0:9 407:3 0:83 2023229:2 0:72 2...,NaN


### 2. Subtree Function 

In [ ]:
def get_subtree_taxids(report_df, target_name):
    names = report_df["name"].astype(str)
    idx = report_df.index[names.str.strip().eq(target_name)]
    if len(idx) == 0:
        return set()
    start = idx[0]
    start_indent = len(re.match(r"^\s*", names.loc[start]).group(0))

    taxids = {int(report_df.loc[start,"taxid"])}

    pos = report_df.index.get_loc(start)

    for i in report_df.index[pos+1:]:
        indent = len(re.match(r"^\s*", names.loc[i]).group(0))
        if indent <= start_indent:
            break
        taxids.add(int(report_df.loc[i, "taxid"]))
    return taxids

In [ ]:
cyano_taxids = get_subtree_taxids(rep, "Cyanobacteria")
len(cyano_taxids), list(sorted(cyano_taxids))[:10]

(29, [1117, 1118, 1129, 1150, 1161, 1185, 1186, 1190, 44887, 52604])

### 3. Filtering Contigs Using 'lca_taxid'
This step assumes that 'df' is the .tsv file with status, contig, and lca_taxid.

In [ ]:
import pandas as pd

df["taxid_extracted"] = (
    df["taxid"]
    .astype(str)
    .str.extract(r"\(taxid\s+(\d+)\)", expand = False)
    .astype("Int64") # nullable integer (allows for NaNs)
)

df[["taxid", "taxid_extracted"]].head(10)

,taxid,taxid_extracted
0,Porphyrobacter sp. CACIAM 03H1 (taxid 2003315),2003315
1,Pseudomonas mendocina NK-01 (taxid 1001585),1001585
2,Gemmatimonas phototrophica (taxid 1379270),1379270
3,Dyella japonica A8 (taxid 1217721),1217721
4,Porphyrobacter sp. CACIAM 03H1 (taxid 2003315),2003315
5,Filimonas lacunae (taxid 477680),477680
6,unclassified (taxid 0),0
7,Caulobacteraceae bacterium OTSz_A_272 (taxid 1...,1759059
8,Brevundimonas sp. LM2 (taxid 1938605),1938605
9,unclassified (taxid 0),0


In [ ]:
cyano = df[df["taxid_extracted"].isin(cyano_taxids)]
len(cyano)

12

In [ ]:
dfC = df[df["status"] == "C"].copy()
dfC["taxid_extracted"] = dfC["taxid_extracted"].astype("int64")

cyano_contigs = dfC[dfC["taxid_extracted"].isin(cyano_taxids)]
cyano_contigs[["contig", "taxid"]].head()
len(cyano_contigs)

(          contig                                          taxid
 457  contig_1420        Synechococcus sp. RCC307 (taxid 316278)
 706  contig_1652     Geminocystis sp. NIES-3709 (taxid 1617448)
 739  contig_1686  Nodularia spumigena UHCC 0039 (taxid 1914872)
 740  contig_1687      Fischerella sp. NIES-3754 (taxid 1752063)
 833   contig_178              Nodularia spumigena (taxid 70799),
 12)

### Regex syntax: 
- r"..." = prefix means raw string (so python doesn't treat backslashes as escape characters), so: 
    - "\n" = newline  
    - r"\n" = "\n" (a literal print).

For '...extract(r"\(taxid\s+(\d+)\)")':
- \( means it is literally looking for a "("  
- taxid means looking for the literal text 'taxid'  
- "\s+" means match one or more whitespace characters (e.g., space, tab, etc.)  
- (\d+): \d = digit; + = one or more, (...) = capture group  
- \) means looking for closed ")"

In [ ]:
# 1) Extract the numeric taxid column for automation (This code may be redundant with earlier steps). 

cyano_contigs = cyano_contigs.copy() 
cyano_contigs["taxid_int"] = (
    cyano_contigs["taxid"].astype(str)
    .str.extract(r"\(taxid\s+(\d+)\)", expand=False)
    .astype("Int64")
)

### 4. Goal: Find Cyanos contigs in MetaBat2 bins and determine what fraction of each bin is cyanobacteria. 

#### Plan: 
1. Build a lookup table: contig -> bin_name by reading headers of each bin FASTA.
2. Join that with your cyano_contigs table 


In [ ]:
from pathlib import Path

metabat_dir = PROJECT / "results/07_bins/KY-Mam-100624-C/metabat2"

# 1) contig -> bin mapping from FASTA headers 
contig2bin = {}

for fa in sorted(metabat_dir.glob("*.fa")): # matches .fa, .fasta, .fa.gz, etc. 
    bin_name = fa.stem # e.g., "bin.1"
    opener = open
    if fa.suffix == ".gz":
        import gzip
        opener = gzip.open
        bin_name = fa.name.replace(".gz", "").split(".fa")[0].split(".fasta")[0]
    
    with opener(fa, "rt") as f: 
        for line in f: 
            if line.startswith(">"):
                contig = line[1:].split()[0] # use first token of header
                contig2bin[contig] = fa.name # could also use bin_name

len(contig2bin), list(contig2bin.items())[:5] 

(1342,
 [('contig_466', 'KY-Mam-100624-C.1.fa'),
  ('contig_1029', 'KY-Mam-100624-C.10.fa'),
  ('contig_1091', 'KY-Mam-100624-C.10.fa'),
  ('contig_1104', 'KY-Mam-100624-C.10.fa'),
  ('contig_1126', 'KY-Mam-100624-C.10.fa')])

In [ ]:
# Annotate Cyanobacteria Contigs: 

cyano_contigs = cyano_contigs.copy()
cyano_contigs["bin"] = cyano_contigs["contig"].map(contig2bin)

cyano_contigs["bin"].value_counts(dropna=False).head(20)

bin
NaN                      9
KY-Mam-100624-C.11.fa    2
KY-Mam-100624-C.5.fa     1
Name: count, dtype: int64

#### Note: if bin is NaN or 'Missing value' these contigs were not binned, which is normal for metaBAT.  
### Get a per-bin summary:

In [ ]:
bin_counts = (
    cyano_contigs.dropna(subset=["bin"]) 
    .groupby("bin")["contig"]
    .nunique()
    .sort_values(ascending=False)
)

# total contigs per bin
all_bins = pd.Series(contig2bin).reset_index()
all_bins.columns = ["contig", "bin"]
total_per_bin = all_bins.groupby("bin")["contig"].nunique()

cyano_per_bin = (
    cyano_contigs.dropna(subset=["bin"])
    .groupby("bin")["contig"].nunique()
)

summary = pd.DataFrame({
    "cyano_contigs": cyano_per_bin,
    "total_contigs": total_per_bin
}).fillna(0)

summary["cyano_frac"] = summary["cyano_contigs"]/ summary["total_contigs"]
summary.sort_values(["cyano_contigs", "cyano_frac"], ascending=False).head(20)

,cyano_contigs,total_contigs,cyano_frac
bin,,,
KY-Mam-100624-C.11.fa,2.0,32,0.062500
KY-Mam-100624-C.5.fa,1.0,222,0.004505
KY-Mam-100624-C.1.fa,0.0,1,0.000000
KY-Mam-100624-C.10.fa,0.0,78,0.000000
KY-Mam-100624-C.12.fa,0.0,1,0.000000
KY-Mam-100624-C.13.fa,0.0,21,0.000000
KY-Mam-100624-C.14.fa,0.0,117,0.000000
KY-Mam-100624-C.15.fa,0.0,6,0.000000
KY-Mam-100624-C.16.fa,0.0,9,0.000000


**Note**: Since many contigs were not binned, we are going to proceed by extracting all cyano genes, binned or not. 
#### Game Plan: 
- Extract all cyano contigs into cyano_contigs.fasta 
- Call genes on that fasta
- Search for KaiABC/photosenthesis/cyanobacteria marker 

#### 1) Define Cyanobacteria Contigs: 
We'll start with text matching because it is faster. A more robust way is to use the cyanobacteria subtree taxids from the kraken report. 

In [ ]:
# 1) Define cyanobacteria contigs
cyano_contigs = df[df["taxid_extracted"].isin(cyano_taxids)].copy()

In [ ]:
# 2) Extract Sequences from the assembly FASTA using pyfaidx 

from pyfaidx import Fasta

asm = str(PROJECT / "results/04_polish/KY-Mam-100624-C.racon2.fasta")
out_fa = str(PROJECT / "results/10_tax_profile/KY-Mam-100624-C/cyano_contigs.fasta")

fa = Fasta(asm, as_raw=True) # uses FASTA index; creates .fai

wanted = set(cyano_contigs["contig"].astype(str))

written = 0

with open(out_fa, "w") as out: 
    for name in fa.keys():
        if name in wanted:
            seq = fa[name]
            out.write(f">{name}\n{seq}\n")
            written += 1

print("Wanted Contigs:", len(wanted))
print("Written Contigs:", written)
print("Output:", out_fa)

Wanted Contigs: 12
Written Contigs: 12
Output: /Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria/GenomicAnalyses/MetagenomicAnalyses/metagenome/results/10_tax_profile/KY-Mam-100624-C/cyano_contigs.fasta


**Note: written << wanted indicates a contig-name mismatch between the kraken output and FASTA headers**

### 2) Use Prodigal to Predict Genes on the Cyanobacteria Contigs 
 Prodigal (Prokaryotic Dynamic Programming Genefinding Algorithm) is a highly well-respected, fast, and accurate, open-source gene prediction tool, particularly for bacterial and archaeal genomes. It is widely used in bioinformatics pipelines, including metagenomics, because it automates the learning of genome properties and consistently ranks among the top-performing gene finders.  
 Prodigal is a commandline tool, but we'll run it in a notebook wrapper below. 

In [45]:
# Subprocess helper for running Prodigal
from pathlib import Path
import shlex

def run_cmd(cmd, cwd=None, env=None):
    """Run a command, stream stdout/sterr, raise on failure."""
    if isinstance(cmd, str):
        printable = cmd
        shell = True
    else:
        printable = " ".join(shlex.quote(str(x)) for x in cmd)
        shell = False 

    print("\n$ " + printable)

    proc = subprocess.Popen(
        cmd, 
        cwd=str(cwd) if cwd else None,
        env=env,
        shell=shell,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    for line in proc.stdout: 
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f"Command failed with exit code {rc}")

In [47]:
# ----- Run Prodigal on Cyanobacteria Contigs ---- 

sample = "KY-Mam-100624-C"
base = PROJECT / "results/10_tax_profile" / sample
inp = base/"cyano_contigs.fasta"
out_faa = base/"cyano_genes.faa"
out_fna = base/"cyano_genes.fna"
out_gff = base/"cyano_genes.gff"

base.mkdir(parents=True, exist_ok=True)

cmd = [
    "prodigal",
    "-i", str(inp),
    "-a", str(out_faa),
    "-d", str(out_fna),
    "-f", "gff",
    "-o", str(out_gff),
    "-p", "meta"
]

run_cmd(cmd)

print("\nDone.")
print("Proteins:", out_faa)
print("Nucleotides:", out_fna)
print("GFF:", out_gff)


$ prodigal -i /Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria/GenomicAnalyses/MetagenomicAnalyses/metagenome/results/10_tax_profile/KY-Mam-100624-C/cyano_contigs.fasta -a /Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria/GenomicAnalyses/MetagenomicAnalyses/metagenome/results/10_tax_profile/KY-Mam-100624-C/cyano_genes.faa -d /Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria/GenomicAnalyses/MetagenomicAnalyses/metagenome/results/10_tax_profile/KY-Mam-100624-C/cyano_genes.fna -f gff -o /Users/jaemac/Desktop/Palmer_Lab/Cyanobacteria/GenomicAnalyses/MetagenomicAnalyses/metagenome/results/10_tax_profile/KY-Mam-100624-C/cyano_genes.gff -p meta
-------------------------------------
PRODIGAL v2.6.3 [February, 2016]         
Univ of Tenn / Oak Ridge National Lab
Doug Hyatt, Loren Hauser, et al.     
-------------------------------------
Request:  Metagenomic, Phase:  Training
Initializing training files...done!
-------------------------------------
Request:  Metagenomic, Phase:  Gene Finding
Finding g

In [48]:
def count_fasta_headers(path):
    n = 0
    with open(path) as f: 
        for line in f:
            if line.startswith(">"):
                n += 1
    return n
print("Predicted proteins:", count_fasta_headers(out_faa))

Predicted proteins: 348


## Next Step: Decide Target Gene Strategy

### Two directions:

#### A) Circadian focus (KaiABC)

- Build curated KaiA/KaiB/KaiC reference set
- Search cyano_genes.faa
- Identify presence/absence
- Map genes back to contigs and bins

#### B) Broader cyanobacteria markers

- Photosystem I/II genes
- rpo genes
- Nitrogen fixation (if relevant)
- Toxin genes (Nodularia relevance?)

## 6) Full-Assembly Gene Prediction with Pyrodigal

### Why search the full assembly — not just cyanobacterial contigs

Kraken2 assigns taxonomy to *contigs* using a lowest-common-ancestor (LCA) approach: it k-merizes the contig and votes across all k-mer hits in the database. A contig containing the KaiABC operon might be pushed to 'unclassified' or to a higher taxonomic level if other genes on the same contig match non-cyanobacterial organisms. This means searching only the 12 Kraken-classified cyanobacterial contigs will miss any Kai genes that landed on mixed-signal or unclassified contigs.

**Solution:** predict genes on all 1,700 contigs, then search the full protein set.

### Why Pyrodigal instead of Prodigal

Prodigal 2.6.3 (the current bioconda version) has a known segfault on very large contigs (≥~500 kb) on arm64 macOS. Pyrodigal is a Python reimplementation of Prodigal that is faster, stable, and produces identical results. Install: `conda install -c bioconda pyrodigal`.

> **Note:** the environment's Python binary is invoked directly (via `cfg['tool_paths']`, see `config.yaml`) rather than through `conda run`, which does not reliably resolve the active environment on all machines.

In [ ]:
# ── Pyrodigal: predict genes on the full assembly ──────────────────────
# Output goes to results/11_genes/<sample>/
# Runtime: ~1–2 min for 1700 contigs on a MacBook Pro M-series

import subprocess
from pathlib import Path

import shutil
from utils import load_config
cfg = load_config()
PYTHON = shutil.which('python3') or cfg.get('tool_paths', {}).get('python3')
sample = 'KY-Mam-100624-C'
asm    = PROJECT / f'results/04_polish/{sample}.racon2.fasta'
out    = PROJECT / f'results/11_genes/{sample}'
out.mkdir(parents=True, exist_ok=True)

script = f'''
import pyrodigal
from pathlib import Path

orf_finder = pyrodigal.GeneFinder(meta=True)
out_dir = Path("{out}")
faa_out = open(out_dir / "all_genes.faa", "w")
fna_out = open(out_dir / "all_genes.fna", "w")
gff_out = open(out_dir / "all_genes.gff", "w")
contig_count = gene_count = 0

with open("{asm}") as f:
    name, seq = None, []
    for line in f:
        line = line.rstrip()
        if line.startswith(">"):
            if name:
                genes = orf_finder.find_genes("".join(seq))
                genes.write_translations(faa_out, sequence_id=name)
                genes.write_genes(fna_out, sequence_id=name)
                genes.write_gff(gff_out, sequence_id=name)
                gene_count += len(genes); contig_count += 1
            name = line[1:].split()[0]; seq = []
        else:
            seq.append(line)
    if name:
        genes = orf_finder.find_genes("".join(seq))
        genes.write_translations(faa_out, sequence_id=name)
        genes.write_genes(fna_out, sequence_id=name)
        genes.write_gff(gff_out, sequence_id=name)
        gene_count += len(genes); contig_count += 1

faa_out.close(); fna_out.close(); gff_out.close()
print(f"Done: {{contig_count}} contigs, {{gene_count}} total proteins")
'''

result = subprocess.run([PYTHON, '-c', script], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## 7) KaiABC Circadian Clock Gene Search (HMMER)

### What are the Kai proteins?

**KaiA, KaiB, and KaiC** form the core oscillator of the cyanobacterial circadian clock — the only biochemically reconstituted circadian clock in bacteria. KaiC is a dual-function ATPase/kinase that phosphorylates and dephosphorylates itself in a ~24-hour cycle. KaiA stimulates KaiC phosphorylation; KaiB sequesters KaiA to allow dephosphorylation. Together they coordinate gene expression to the light/dark cycle.

- **KaiC** is the most conserved and is found in many cyanobacteria and some Archaea/Rhizobiaceae.
- **KaiB** is moderately conserved across cyanobacteria.
- **KaiA** is the least conserved and is absent in some cyanobacterial lineages (*Prochlorococcus*, some *Synechococcus*).

### HMM profiles used

We use profiles from the NCBI PGAP database (Jan 2026 release). **Important:** the commonly cited TIGRFAM accessions TIGR02617/18/19 now refer to *unrelated* proteins in the current database (tryptophanase, etc.). The correct accessions are:

| Gene | Profile | Source | Notes |
|------|---------|--------|---------|
| KaiC | TIGR02655.1 | NCBI PGAP (JCVI) | Cyanobacterial equivalog, 484 aa model |
| KaiB | TIGR02654.1 | NCBI PGAP (JCVI) | Cyanobacterial equivalog, 87 aa model |
| KaiA (C-term) | PF07688 | Pfam / InterPro | Domain model, Bacteria only |
| KaiA (N-term) | PF21714 | Pfam / InterPro | Domain model |

Profiles are stored in `db/kai_hmm/`. The combined pressed database is `KaiABC_correct.hmm`.

### Why search all proteins, not just cyanobacterial ones?

As noted in Section 6, KaiABC genes may reside on contigs that Kraken2 did not classify as cyanobacterial. A whole-assembly protein search is more comprehensive.


In [ ]:
# ── Step 1: Download HMM profiles (run once) ─────────────────────────
# Profiles are already at: db/kai_hmm/KaiABC_correct.hmm
# Re-run this cell only if that file is missing.

import subprocess
from pathlib import Path

hmm_dir = PROJECT / 'db' / 'kai_hmm'
hmm_dir.mkdir(parents=True, exist_ok=True)

cfg = load_config()
HMMPRESS  = shutil.which('hmmpress') or cfg.get('tool_paths', {}).get('hmmpress')

profiles = {
    'TIGR02655.1.HMM': 'https://ftp.ncbi.nlm.nih.gov/hmm/current/hmm_PGAP.HMM/TIGR02655.1.HMM',
    'TIGR02654.1.HMM': 'https://ftp.ncbi.nlm.nih.gov/hmm/current/hmm_PGAP.HMM/TIGR02654.1.HMM',
}
pfam_profiles = {
    'PF07688_KaiA.hmm': 'https://www.ebi.ac.uk/interpro/wwwapi//entry/pfam/PF07688?annotation=hmm&download',
    'PF21714_KaiA.hmm': 'https://www.ebi.ac.uk/interpro/wwwapi//entry/pfam/PF21714?annotation=hmm&download',
}

combined = hmm_dir / 'KaiABC_correct.hmm'
if not combined.exists():
    import urllib.request, gzip, shutil
    for fname, url in profiles.items():
        out = hmm_dir / fname
        if not out.exists():
            urllib.request.urlretrieve(url, out)
            print(f'Downloaded {fname}')
    for fname, url in pfam_profiles.items():
        gz = hmm_dir / (fname + '.gz')
        out = hmm_dir / fname
        if not out.exists():
            urllib.request.urlretrieve(url, gz)
            with gzip.open(gz, 'rb') as fi, open(out, 'wb') as fo:
                shutil.copyfileobj(fi, fo)
            gz.unlink()
            print(f'Downloaded {fname}')
    # Combine and press
    with open(combined, 'wb') as out_f:
        for p in sorted(hmm_dir.glob('*.hmm')) + sorted(hmm_dir.glob('*.HMM')):
            if p != combined:
                out_f.write(p.read_bytes())
    subprocess.run([HMMPRESS, str(combined)], check=True)
    print('Combined HMM database ready.')
else:
    print('HMM database already exists:', combined)

In [ ]:
# ── Step 2: Run hmmsearch against all 75,230 proteins ────────────────

cfg = load_config()
HMMSEARCH = shutil.which('hmmsearch') or cfg.get('tool_paths', {}).get('hmmsearch')
sample    = 'KY-Mam-100624-C'
faa       = PROJECT / f'results/11_genes/{sample}/all_genes.faa'
hmm_db    = PROJECT / 'db/kai_hmm/KaiABC_correct.hmm'
out_dir   = PROJECT / f'results/11_genes/{sample}'

tblout  = out_dir / 'kai_hits.tblout'
domtbl  = out_dir / 'kai_hits.domtblout'

subprocess.run([
    HMMSEARCH,
    '--tblout', str(tblout),
    '--domtblout', str(domtbl),
    '-E', '1e-5', '--cpu', '4',
    str(hmm_db), str(faa)
], check=True, capture_output=True)

# Parse hits
hits = []
with open(tblout) as f:
    for line in f:
        if line.startswith('#') or not line.strip():
            continue
        parts = line.split()
        hits.append({'protein': parts[0], 'profile': parts[2],
                     'evalue': float(parts[4]), 'score': float(parts[5])})

import pandas as pd
hits_df = pd.DataFrame(hits)
if len(hits_df):
    display(hits_df)
else:
    print('No hits at E-value <= 1e-5')

## 8) Coverage of Cyanobacterial Contigs

To interpret gene search results, we need to know whether the absence of KaiABC could be explained by *low sequencing depth* (i.e., the genes exist but weren't assembled well), or whether the cyanobacterial contigs themselves are well-covered.

We use the self-sample BAM from `results/06_cov/<sample>/maps/<sample>.bam` — this is the BAM of a sample's own reads mapped back to its own assembled contigs, generated during the differential coverage step.


In [ ]:
# ── Mean coverage per cyanobacterial contig ──────────────────────────

import subprocess
import pandas as pd

cfg = load_config()
SAMTOOLS = shutil.which('samtools') or cfg.get('tool_paths', {}).get('samtools')
sample   = 'KY-Mam-100624-C'
bam      = PROJECT / f'results/06_cov/{sample}/maps/{sample}.bam'

# Cyanobacterial contig names from Kraken2 classification
cyano_contigs_list = [
    'contig_1420', 'contig_1652', 'contig_1686', 'contig_1687', 'contig_178',
    'contig_225',  'contig_227',  'contig_228',  'contig_410',  'contig_427',
    'contig_531',  'contig_800'
]

# Stream samtools depth and compute per-contig mean
proc = subprocess.run(
    [SAMTOOLS, 'depth', '-a', str(bam)],
    capture_output=True, text=True
)

cyano_set = set(cyano_contigs_list)
depth_sum = {}; depth_cnt = {}
for line in proc.stdout.splitlines():
    ctg, pos, dep = line.split('\t')
    if ctg in cyano_set:
        depth_sum[ctg] = depth_sum.get(ctg, 0) + int(dep)
        depth_cnt[ctg] = depth_cnt.get(ctg, 0) + 1

rows = [{'contig': c, 'length_bp': depth_cnt.get(c, 0),
         'mean_depth': round(depth_sum.get(c, 0) / max(depth_cnt.get(c, 1), 1), 1)}
        for c in cyano_contigs_list]
cov_df = pd.DataFrame(rows).sort_values('mean_depth', ascending=False)
display(cov_df)

low = cov_df[cov_df['mean_depth'] < 10]
print(f'\n{len(low)}/{len(cov_df)} contigs have < 10x mean depth (fragmented/low-abundance organisms)')

> ## ⚠️ CORRECTED — see FLAG 4 in `README.md`
>
> The KaiABC absence reported below is real, but the interpretation is not. The three
> "well-covered cyanobacterial contigs" searched here are **green algal chloroplast
> genomes** (contig_1686 → *Mychonastes* plastid; contig_1652 → *Chlorella* plastid; see
> §13). **Plastids do not carry kaiABC** — the circadian oscillator was lost during
> endosymbiotic genome reduction and the residual clock in algae is nuclear-encoded. Finding
> zero Kai hits in a chloroplast genome is the expected result, not evidence of cave-adapted
> gene loss. The cave/relaxed-selection hypothesis below is **not supported by these data**;
> testing it would require contigs from an actual free-living cyanobacterium.

## 9) Findings & Interpretation — KY-Mam-100624-C (Mammoth Cave)

### Key results

| Gene | Result | Notes |
|------|--------|---------|
| KaiA | **Absent** | No hits at any threshold across 75,230 proteins |
| KaiB | **Absent** | No hits at any threshold |
| KaiC | **3 hits (E ≤ 1e-5)** | All in Rhizobiaceae, not cyanobacteria |

The three KaiC hits are in Alphaproteobacteria (Sinorhizobium, Mesorhizobium, Rhizobium) — this is biologically real. Rhizobiales are known to carry KaiBC-like oscillator proteins. The absence of KaiA is expected (Rhizobiales don't have it).

### Why no cyanobacterial KaiABC?

Coverage analysis shows a clear split among the 12 Kraken-classified cyanobacterial contigs:

- **3 well-covered contigs (68–95x):** *Nodularia spumigena* (78 kb), *Fischerella* sp. (42 kb), *Geminocystis* sp. (114 kb)
- **9 low-coverage contigs (<15x):** likely represent rare/low-abundance organisms

The 3 high-coverage contigs yielded **174 predicted proteins**. Even at E-value=1 with 6 different HMM profiles (including the broad KaiC ATPase domain PF06745), **zero Kai-related hits** were found. This is not an assembly or depth artifact for those organisms.

### Biological interpretation — the cave context

> **KY-Mam-100624-C is from a Mammoth Cave river — a permanent dark environment.**

Circadian clocks are driven by and synchronized to light/dark cycles. In a cave with no photocycles, the selective pressure to maintain a functional circadian oscillator is greatly reduced or absent. Potential explanations for KaiABC absence:

1. **True gene loss:** cave-adapted cyanobacteria may have lost KaiABC through relaxed selection — this would be a publishable observation.
2. **Extreme sequence divergence:** the Kai proteins may have diverged beyond HMM detection thresholds. A BLAST search against the specific reference genomes (*Nodularia spumigena* CCY9414, *Fischerella* PCC 7521) would test this.
3. **Incomplete genome coverage:** the 78–114 kb contigs represent partial genomes; the KaiABC operon could simply be on unassembled portions.

### Recommended follow-up

1. **BLASTp** the 174 high-coverage cyanobacterial proteins against known KaiABC from close relatives (*Nodularia spumigena* CCY9414, *Fischerella muscicola* PCC 7414) — this will catch diverged sequences the HMM misses.
2. **Compare with surface-water samples** (FL-Wim, TN-Cla, GA-Cor) — if those samples show KaiABC, the cave-specific absence becomes a strong finding.
3. **Read-level Bracken** analysis to estimate what fraction of reads map to cyanobacteria across all 10 samples — gives context on cyanobacterial abundance before diving into gene-level analysis.


## 10) Next Steps: Batch Pipeline for Remaining 9 Samples

All 10 samples currently have:
- ✅ Flye assembly (`results/03_assembly/`)
- ✅ Racon × 2 polishing (`results/04_polish/`)
- ❌ Medaka polishing — **not yet run for any sample**

> **Important:** `META_CONTIGS` in the Snakefile currently points to the racon2 output (marked as TEMP). Before batching, decide whether to run Medaka first and update `META_CONTIGS` to use `medaka_consensus`. Medaka provides higher per-base accuracy which improves downstream gene calling and binning.

### Pipeline improvements to make before batching

1. **Run Medaka** on at least the priority samples, then update `META_CONTIGS`.
2. **Add CheckM2** for bin quality assessment — currently absent from the Snakefile. This is required before calling any bin a MAG.
3. **MaxBin2 + DAS_Tool** are disabled on macOS but can be re-enabled when running on a Linux server for better bin reconciliation.

### Recommended batch order

Run surface-water samples first for KaiABC comparison with the cave sample:
```python
# Highest-priority samples (largest FASTQs = most data)
priority = ['FL-Wim-100124-B', 'TN-Cla-092324-E', 'GA-Cor-092624-B']
```


In [ ]:
# ── Batch run template for remaining samples ─────────────────────────
# Uncomment and adjust as needed.

# Priority: surface-water samples for KaiABC comparison with cave sample
# focus_samples = ['FL-Wim-100124-B', 'TN-Cla-092324-E', 'GA-Cor-092624-B']

# Or all remaining samples:
# focus_samples = [
#     'MN-Por-091024-A', 'TN-Cla-092324-E', 'WI-Bar-090824-B',
#     'NC-Che-092424-E', 'KY-Mam-100624-B', 'GA-Cor-092624-B',
#     'FL-Wim-100124-B', 'MN-Col-091924-B', 'FL-Hom-052325-G'
# ]

# Run through Kraken2 taxonomy for each
# snakemake(
#     targets  = ['all'],
#     cores    = 6,
#     use_conda = True,
#     focus_samples    = focus_samples,
#     cov_read_samples = focus_samples,   # or include KY-Mam samples for cross-coverage
# )

print('Batch template ready — uncomment the block above to run.')


## 11) Reference Genome Mapping & KaiABC Absence (3-Method Proof)

Three independent methods all confirm that KaiABC circadian clock genes are undetectable
in the cave cyanobacterial sequences from **KY-Mam-100624-C** (Mammoth Cave river, permanent dark).

### Method 1 — HMMER profile search

Six HMM profiles (KaiA PF11640, KaiB PF07689, KaiC ATPase domain PF06745, and three
supporting profiles) were run against all 348 proteins predicted from the 12 cyano contigs,
using an inclusive E-value threshold of 1 (i.e., accept even marginal hits).

**Result: 0 hits.**

### Method 2 — BLASTp against all assembly proteins

16 KaiABC reference proteins (from *Nodularia spumigena* UHCC0039, *Fischerella muscicola*
PCC7414, and *Geminocystis* sp. NIES-3709) were searched against the full-assembly protein
set (75,230 proteins, all contigs including non-cyanobacterial).

- **Query:** `db/ref_genomes/` KaiABC protein sequences (16 queries)
- **Database:** all predicted proteins from the KY-Mam-100624-C assembly
- **E-value threshold:** 1×10⁻³

**Result: 0 hits in any of the 12 cyano contigs.**

The only hits were in non-cyanobacterial contigs:
- `contig_85` → *Sinorhizobium* sp. (Rhizobiaceae), 25–40% identity
  - Consistent with distant ATPase domain homology; Rhizobiales carry KaiBC-like oscillators.
  - This confirms the BLAST search was functioning — it can find distant homologs when present.

### Method 3 — Read-level mapping to KaiABC locus

All 161,253 filtered ONT reads from KY-Mam-100624-C were mapped (minimap2 `map-ont`) directly
to the KaiABC genomic locus in two reference genomes, bypassing assembly entirely.

**Reference locus — *Nodularia spumigena* UHCC0039 (NZ_CP020114.1):**

| Gene | Coordinates |
|------|------------|
| KaiA | 1,971,121 – 1,971,489 |
| KaiB | 1,972,937 – 1,973,251 |
| KaiC | 1,973,343 – 1,974,908 |
| Window used | 1,961,121 – 1,984,908 (23 kb) |

**Results:**
- Reads mapping to the **23 kb KaiABC window**: **0**
- Reads mapping to **other parts of the Nodularia chromosome**: **6,847** (58–234× depth)
- Reads mapping to **Geminocystis NIES-3709 KaiABC locus**: **0**

The 6,847 reads mapping elsewhere on the Nodularia chromosome confirm the pipeline works
and that the cave sample contains a *Nodularia*-related organism. The KaiABC region shows a
clear gap in an otherwise covered chromosome — this is not a mapping artifact.

### Summary

| Method | Input | Result |
|--------|-------|--------|
| HMMER (6 profiles, E=1) | 348 cyano proteins | 0 hits |
| BLASTp (E=1×10⁻³) | 75,230 all-assembly proteins | 0 hits in cyano contigs |
| Read mapping (map-ont) | 161,253 filtered reads vs KaiABC locus | 0 reads at locus |

**Conclusion:** KaiABC is absent from the sequenced cave cyanobacteria by all three methods.
The absence is not attributable to low coverage, assembly failure, or search sensitivity.

### Reference genomes

Stored in `db/ref_genomes/`:
- *Nodularia spumigena* UHCC0039 — GCF_003054475.1
- *Fischerella muscicola* PCC7414 — GCF_000317205.1
- *Geminocystis* sp. NIES-3709 — GCF_001548115.1

Results in `results/12_ref_mapping/`.


## 12) KY-Mam-B Comparison (Outside Cave)

**KY-Mam-100624-B** is the paired sample collected *outside* the cave at Mammoth Cave National
Park — a surface site exposed to normal light/dark cycles. Running the same KaiABC locus mapping
on this sample tests whether KaiABC absence is cave-specific or a property of the local cyanobacterial lineage.

### Read mapping to Nodularia UHCC0039

| Mapping target | KY-Mam-100624-C (cave) | KY-Mam-100624-B (surface) |
|----------------|------------------------|---------------------------|
| Full *Nodularia* reference (whole chromosome) | 6,847 reads | 6,149 reads |
| KaiABC locus (23 kb window) | **0 reads** | **0 reads** |
| *Geminocystis* KaiABC locus | **0 reads** | **0 reads** |

Both samples map a similar number of reads to the *Nodularia* chromosome (~6,000–7,000),
confirming comparable sequencing depth and that the same cyanobacterial lineage is present
at both sites. Neither shows any reads at the KaiABC locus.

### Interpretation

KaiABC absence is shared by the inside-cave *and* outside-cave Mammoth Cave samples.
This means:

1. **Not a cave-specific adaptation** — the absence does not arise from living in permanent darkness.
2. **A lineage property** — the cave and surface cyanobacteria at MCNP likely represent the
   same novel clade, one that diverged from KaiABC-containing relatives before the cave/surface split.
3. **Wider comparison needed** — the key question now is whether samples from other geographic
   sites (FL-Wim, TN-Cla, GA-Cor) show KaiABC. If those surface-water samples also lack KaiABC,
   the absence is a clade-wide character. If they have KaiABC, the absence is specific to the
   MCNP/Mammoth Cave lineage.

> **Next action:** run the same 3-method KaiABC screen on FL-Wim-100124-B, TN-Cla-092324-E,
> and GA-Cor-092624-B to resolve this question.


> ## ⚠️ CORRECTED — see FLAG 4 in `README.md`
>
> The contigs this section calls novel cave cyanobacteria are **green algal chloroplast
> genomes**. The 16S BLAST below was run against NCBI nt **with a Cyanobacteria filter**.
> In NCBI taxonomy a chloroplast sequence is filed under its eukaryotic host, not under
> Cyanobacteria, so that filter makes a chloroplast hit *impossible to return* — every
> plastid query is forced onto its nearest free-living cyanobacterial relative, and the
> resulting 94–99% identity reads as a novel lineage.
>
> Plastid-vs-cyanobacterium was the known open question for these contigs; this section is
> where a search that **could not express "plastid" as an answer** converted that open
> question into a positive identification. The analysis below is kept for the record, with
> corrections inline.

## 13) 16S rRNA Phylogenetics & Species Novelty

### Barrnap 16S predictions

**Barrnap** found 16S rRNA genes in 6 of the 12 Kraken-classified "cyano" contigs.

| Contig | Contig length | 16S | Note |
|--------|---------------|-----|------|
| contig_1652 | 114,255 bp | 1,490 bp, single copy | |
| contig_1686 | 78,005 bp | 1,484 bp **× 2, opposite orientations, 51 kb apart** | called a "duplicated ribosomal operon" below — it is a plastid **inverted repeat** |
| contig_225 | 46,945 bp | 1,541 bp | |
| contig_227 | 72,723 bp | 1,541 bp **× 2, opposite orientations, at both contig ends** | inverted repeat spanning the circular junction |

### ~~NCBI BLAST results (nt database, Cyanobacteria filter)~~ — invalid, see banner

| Contig | Kraken2 assignment | 16S BLAST top hit (Cyano-filtered) | Identity |
|--------|--------------------|-------------------|----------|
| contig_1652 | *Geminocystis* | ~~*Loriellopsis* sp. BFS1~~ | ~~98–99%~~ |
| contig_1686 | *Nodularia spumigena* | ~~Uncultured cyanobacterium Dpcom212/Dpcom152~~ | ~~100%~~ |
| contig_225 | Oscillatoriales JSC-12 | ~~Oscillatoriales cyanobacterium HF1~~ | ~~94–97%~~ |
| contig_227 | Oscillatoriales JSC-12 | ~~Oscillatoriales cyanobacterium HF1~~ | ~~94–97%~~ |

**Kraken2 was substantially wrong for all organisms** — that part stands, and for a reason
neither Kraken2 nor the filtered BLAST could reveal: none of these are cyanobacteria.

### Corrected assignments — unfiltered SILVA SSU NR99

| Contig | SILVA top hit | Identity | Aln |
|--------|---------------|----------|-----|
| contig_1652 | ***Chlorella pyrenoidosa*** chloroplast (ANZC01004531) | **98.7%** | 1,473 bp |
| contig_1686 | ***Mychonastes jurisii*** chloroplast (KT625411) | **99.7%** | 1,484 bp |
| contig_225 / 227 | Chloroplast, unidentified host (FPLM01003032) | 93.6% | 1,523 bp |
| contig_1637 | *Micractinium conductrix* / *Auxenochlorella pyrenoidosa* chloroplast | 99.8% | — |

Every one of these sits in `Bacteria;Cyanobacteriota;Cyanobacteriia;Chloroplast` — SILVA's
chloroplast bin, which is filed under Cyanobacteriota because plastids **descend from**
cyanobacteria. It records evolutionary origin, not free-living status.

**Why contig_1686 hit "uncultured cyanobacterium Dpcom212" at 100%:** those 2003 GenBank
clones are themselves plastid sequences deposited under a cyanobacterial name. In SILVA they
sit in the chloroplast bin too (99.0%, `…;Chloroplast;…;uncultured bacterium`). The 100%
identity was a match to another mislabelled plastid.

### Three independent lines of evidence

1. **Phylogeny** (`results/14_phylo_16S/all_16S.treefile`; IQ-TREE, GTR+F+I+R2, 1000 UFBoot,
   46 taxa including 15 chloroplast and 17 cyanobacterial references). All six query contigs
   fall **inside the Chlorophyta plastid radiation**, not merely near it:
   - (contig_225, contig_227) 99/100, + contig_427 → 81 UFBoot
   - (contig_531, contig_3345) 98.7/100, sister to ***Desmodesmus abundans*** chloroplast at
     95.7/100
   - (contig_1504, contig_3677) 92/100
   - the whole group + *Scenedesmus obliquus* / *Coelastrella saipanensis* plastids: **100/100**
   - plastid monophyly including *Cyanophora*, *Porphyra* and *Euglena*: **97.1/100**, sister
     to the free-living cyanobacteria
   *Loriellopsis cavernicola* (NR_117881.1) is now **in** this tree and no query contig is
   anywhere near it — it groups with *Chroococcidiopsis*, well inside the cyanobacteria.
2. **Genomic context** (`03a_contig_id_gene_context.ipynb` §3) — `Bacterial-like hits: 0 |
   Plastid/eukaryotic hits: 3` for both samples. Genes flanking the 16S are psbC (PSII 44 kDa,
   92–96% id), atpF/atpH (ATP synthase CF0, 92–99% id) and plastid LAGLIDADG intron ORFs.
3. **Genome architecture** — contig_1686 and contig_227 each carry the 16S in **two
   opposite-orientation copies** tens of kb apart. That is the chloroplast inverted repeat.
   Cyanobacterial chromosomes are 2–9 Mb and do not have this structure.

### What the 94–97% identity actually meant

Below the ~94.5% genus threshold (Yarza et al. 2014) reads as genus-level novelty **only for
comparisons within free-living bacteria**. A plastid 16S forced against cyanobacterial
references lands at 90–95% by construction — it is measuring the age of primary endosymbiosis,
not the novelty of a cave organism.

### FastTree phylogeny (superseded)

The original tree used only references returned by the Cyano-filtered BLAST, so it contained
no chloroplast sequences and could not have placed these contigs correctly. Superseded by
`03b_16S_phylogeny.ipynb` → `results/14_phylo_16S/all_16S.treefile`.


In [ ]:
# Read and display tree (visualize in iTOL: https://itol.embl.de or FigTree)
from pathlib import Path
tree_nwk = PROJECT / 'results/12_ref_mapping/tree_16S.nwk'
print(tree_nwk.read_text())
print("\nTo visualize: upload tree_16S.nwk to https://itol.embl.de")
print("or open in FigTree (free download)")


> ## ⚠️ CORRECTED — see FLAG 4 in `README.md`
>
> The completeness and ANI numbers below are correct, and the cyanobacterial genome size used
> as the denominator was a **deliberate fixed benchmark** — applied uniformly across samples so
> recovery was comparable, at a time when whether these were cyanobacteria was the open
> question. Nothing here was measured wrong. What has changed is that the identity is now
> settled, which adds a second reading. These contigs are **green algal chloroplast genomes**, which run
> 103–204 kb in this lineage (*Mychonastes jurisii* 103 kb, *Chlorella vulgaris* 151 kb,
> *Scenedesmus obliquus* 161 kb, *Coelastrella saipanensis* 196 kb, *Chlamydomonas
> reinhardtii* 204 kb). Read against plastid genomes rather than 5–8 Mb cyanobacterial
> chromosomes, the assemblies are not 1–2% fragments — they are **substantially complete
> organelle genomes**:
>
> | Contig | Assembled | Matched plastid genome | Approx. completeness |
> |---|---|---|---|
> | contig_1652 | 114 kb | *Chlorella* cp, ~151 kb | ~75% |
> | contig_1686 | 78 kb | *Mychonastes jurisii* cp, 103 kb | ~76% |
> | contig_225 + contig_227 | 120 kb (2 contigs) | Chlorophyceae cp, 155–196 kb | ~65–75% |
>
> The FastANI result (~74–76% against three cyanobacterial genomes) is likewise exactly what
> a chloroplast genome gives against free-living cyanobacteria. It does **not** indicate a
> novel cyanobacterial genus. The "at least 3 novel lineages" conclusion at the end of this
> section does not hold.

## 14) Genome Completeness & FastANI

### Completeness estimates

The 12 cyano contigs represent small fragments of full cyanobacterial genomes. Coverage on
assembled contigs is good (68–95×), but cyanobacteria are low-abundance in this community
(~1% of reads), limiting how much genome can be assembled from a single run.

| Organism | Assembled (this study) | Typical genome size | **Estimated completeness** |
|----------|----------------------|--------------------|--------------------------|
| *Loriellopsis*-like (contig_1652) | ~114 kb | 6–8 Mb | ~1.5–2% |
| Dpcom uncultured (contig_1686) | ~78 kb | 5–7 Mb | ~1–1.5% |
| Oscillatoriales (contig_225/227) | ~120 kb (2 contigs) | 5–7 Mb | ~2% |
| Fischerella-like (remaining contigs) | ~42 kb | 7–13 Mb | <1% |

These are substantially incomplete genomes assembled from shotgun metagenomics at ~1%
cyanobacterial abundance. To improve:
- **Deeper sequencing** of the same samples (e.g., 10× more reads per sample)
- **Enrichment + re-sequencing** (filter cells, lyse, amplify) targeted at cyanobacteria
- **Long-read co-assembly** across multiple samples (KY-Mam-B + KY-Mam-C together)

### FastANI analysis

FastANI v1.34 (fragment length 500 bp) was run with the cave contigs as queries against
all three reference genomes.

| Query | Reference | ANI |
|-------|-----------|-----|
| All cave cyano contigs | *Nodularia* UHCC0039 | ~74–76% |
| All cave cyano contigs | *Fischerella* PCC7414 | ~74–76% |
| All cave cyano contigs | *Geminocystis* NIES-3709 | ~74–76% |

**Interpretation of ANI thresholds:**

| ANI range | Taxonomic level |
|-----------|-----------------|
| ≥95% | Same species |
| 88–95% | Same genus (approximate) |
| 80–88% | Same family (approximate) |
| <80% | Different genus or higher |

All cave sequences fall at ~74–76% ANI vs all three references — well below genus level.
This is fully consistent with the 16S result: these organisms are genuinely novel lineages,
distantly related to any sequenced cyanobacterium.

> **Caveat:** FastANI requires >10% genome coverage (ideally >20%) for reliable estimates.
> Our assemblies represent ~1–2% of the genome. The reported ANI values are lower bounds
> on divergence — they confirm deep divergence but the exact value is uncertain. The qualitative
> conclusion (different genus or higher) is robust.

### Overall summary

The KY-Mam-100624-C cave cyanobacteria comprise at least 3 novel lineages:

1. A *Loriellopsis*-related Nostocales (Symphyonemataceae), known from caves — expected.
2. The first genome for an unnamed Dpcom clade, previously known only from a 2003 soil clone — novel.
3. A deeply diverged Oscillatoriales with no named relative above 94% 16S identity — potentially a new genus.

None carry detectable KaiABC. Whether this is a lineage-wide character shared with other
MCNP samples (KY-Mam-B) and geographically distant samples remains the key open question.


## Running KY-Mam-100624-B 

In [ ]:
# In Terminal run:

# Make sure you're in the correct directory and conda environment (genomics_arm64) before executing Snakemake.

# FocusSAMPLES="KY-Mam-100624-B" \
# COV_READ_SAMPLES="KY-Mam-100624-B,KY-Mam-100624-C" \
# snakemake --cores 8 --use-conda
